# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/jamal-ai-dev/flyrank-ml-internship.git
import pandas as pd
df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
df.head()

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane (or freestyle) and why

*I'm choosing Lane 2 — Refresh / Content Opportunity Scoring. This lane asks which content pages should be reviewed first for refresh, out of limited review capacity. The starter pipeline already showed a random forest model beating a hand-written baseline rule on this exact task (Precision@50 improved from 0.24 to 0.74), which tells me there's real, learnable signal in this data — not just noise. This also lines up with what I want my portfolio to demonstrate: a model that measurably beats a simple rule, and that I can explain in plain language.*

## 2. The question: decision, action, cost of a wrong call

*Decision: Which content pages should a content team prioritize for review first, given limited time?
Unit of analysis: One content page, aggregated over a trailing 90-day window.
Action: A content reviewer opens the highest-ranked pages and decides whether to refresh, expand, or leave each one alone.
Cost of a wrong call: If a page is flagged but wasn't actually declining, the team wastes limited review time that could've gone to a page that really needed it. If a genuinely declining, high-traffic page is missed, it keeps losing search visibility unnoticed.
Why ML helps: With thousands of pages and one reviewer's limited time, some ranking is unavoidable — the real question is whether a learned ranking does it more accurately than a fixed rule. The starter pipeline's baseline-vs-model comparison suggests yes.*

## 3. Quick look at the data (2-3 real numbers)

*54.2% of all pages in the dataset are labeled as declining, and those pages account for 51.3% of total search impressions (79,994,363 of 156,010,989) — meaning search visibility loss isn't a minor edge case, it's roughly half the traffic. Even after narrowing to only high-traffic declining pages (impressions_90d ≥ 500), there are still 9,961 candidates — far more than a reviewer can manually check, which is exactly the prioritization problem this lane is built to solve.*

In [2]:
declining_pct = (df['trend_direction'] == 'down').mean() * 100
declining_impressions = df.loc[df['trend_direction'] == 'down', 'impressions_90d'].sum()
total_impressions = df['impressions_90d'].sum()
high_traffic_declining = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)]

print(f"Declining pages: {declining_pct:.1f}%")
print(f"Impressions on declining pages: {declining_impressions:,} of {total_impressions:,} "
      f"({declining_impressions/total_impressions*100:.1f}%)")
print(f"High-traffic declining pages: {len(high_traffic_declining):,}")

Declining pages: 54.2%
Impressions on declining pages: 79,994,363 of 156,010,989 (51.3%)
High-traffic declining pages: 9,961


## 4. Careful words: what I can and can't claim

This analysis can say which pages are observed to have declining search visibility, and can rank pages by how strongly they resemble past pages that needed review, using transparent signals. It's decision-support for a human reviewer, not a guarantee of outcome.
This analysis cannot claim that refreshing a page will cause it to recover — that would require a controlled experiment, not historical patterns. It also never claims to predict or model Google's actual ranking algorithm; trend_direction and trend_pct are only ever used as the label being explained, never as input features.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.